In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import kagglehub

# Download latest version
path = kagglehub.dataset_download("oddrationale/mnist-in-csv")

print("Path to dataset files:", path)






Path to dataset files: /kaggle/input/datasets/oddrationale/mnist-in-csv


Goal here is to train many different classifiers, and see if the ensemble is a better predictor
Then, I will train my own blender from the results of the ensemble, comparing results again.

In [2]:
train = pd.read_csv(path + "/mnist_train.csv")
test = pd.read_csv(path + "/mnist_test.csv")

In [3]:
X_train = train.iloc[:, 1:].to_numpy()
y_train = train.iloc[:, 0].to_numpy()

X_test = test.iloc[:, 1:].to_numpy()
y_test = test.iloc[:, 0].to_numpy()

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train,test_size = 10000,random_state = 42)

In [5]:
#random forest, extra forest, gradient boost, svm, ada boost   so 5 models total - 5/31 removed the boost classifiers  so 3 total
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC

rfc = RandomForestClassifier(random_state = 42)
etc = ExtraTreesClassifier(random_state = 42)


svmc = SVC(kernel = "rbf", random_state = 42)

frstmodels = [rfc,etc]

In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [7]:

# optimal_models = []
# #using gridsearch to find optimal hyperparamers

# param_frst = {"n_estimators": [300,400],
#     "max_depth": [20,30],
#     "min_samples_split": [2,3],
#     "min_samples_leaf": [1,2,3],
#     "max_features": ["sqrt"]}
# param_svm = {
#     "svc__C": [10],
#     "svc__gamma": [ 0.001],
    
# }


# for frst in frstmodels:
#     gridsearch = GridSearchCV(frst, param_frst, cv = 3, n_jobs=-1)
#     optimal_models.append(gridsearch)

# gridsearch = GridSearchCV(svm_pipeline, param_svm,cv = 3, n_jobs = -1)
# optimal_models.append(gridsearch)

In [8]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", svmc)
])
svm_pipeline.set_params(
    svc__C=10,
    svc__gamma=0.001,
    svc__kernel="rbf"
)
forest = RandomForestClassifier(max_depth= 20, max_features= "sqrt", min_samples_leaf= 1,
                                      min_samples_split= 2, n_estimators= 400, random_state = 42)
extratrees = ExtraTreesClassifier(max_depth= 30, max_features= "sqrt", min_samples_leaf= 1,
                                       min_samples_split= 2, n_estimators= 300, random_state = 42 )
SVMC =svm_pipeline

optimal_models = [forest,extratrees,SVMC]

In [9]:
model_names = ["forest", "extratrees", "svmc"]

fitted_models = []
fitted_base_estimators = []

for name, model in zip(model_names, optimal_models):
    model.fit(X_train, y_train)

    fitted_models.append(model)
  

   
    print("Validation accuracy:", model.score(X_val, y_val))

Validation accuracy: 0.9698
Validation accuracy: 0.9731
Validation accuracy: 0.9717


In [10]:
import joblib
from pathlib import Path

model_dir = Path("/kaggle/working/models")
model_dir.mkdir(exist_ok=True)

joblib.dump(fitted_models, model_dir / "best_models.joblib")
joblib.dump(fitted_base_estimators, model_dir / "manual_base_estimators.joblib")

['/kaggle/working/models/manual_base_estimators.joblib']

5/25 I don't know why the SVC is so bad, but I haven't learned much about them so I will just pop it

5/31 The reason was that there was no scaling on the data, so after adding that it should be fine

5/31 Realized that boosting models are too slow to train for this dataset

In [11]:
predictions = []
for model in fitted_models:
    preds = model.predict(X_val)
    predictions.append(preds)

print (predictions)

[array([7, 3, 8, ..., 9, 8, 1], shape=(10000,)), array([7, 3, 8, ..., 9, 8, 1], shape=(10000,)), array([7, 3, 8, ..., 9, 8, 1], shape=(10000,))]


In [12]:
from scipy.stats import mode
preds = np.array(predictions).astype(int)
ensemble_preds = mode(preds, axis = 0).mode
print(ensemble_preds)

[7 3 8 ... 9 8 1]


In [13]:
from sklearn.metrics import accuracy_score

print(accuracy_score(ensemble_preds,y_val))

0.9739


First ensemble fitting got accuracy of 90.25%, however the SVMC got 94%, and is carrying the random forests, so I am retraining the random forests by increasing max depth. which was way too low, at 7. 

89 and 87 for rndfrs, extrfrst


RandomForestClassifier
{'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}
Score: 0.9633

ExtraTreesClassifier
{'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Score: 0.9672

Pipeline - svmc
{'svc__C': 10, 'svc__gamma': 0.001}
Score: 0.9608

5/31 96.84% accuracy now with ensemble, higher score than the highest individual classifier

5/31 Now I will train a blender, so that this is a stacking ensemble.
Feeding the results from the predictions into another classifier

In [14]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

a = np.array(predictions[0])
b = np.array(predictions[1])
c = np.array(predictions[2])

combined = np.column_stack((a,b,c))
print(combined)

[[7 7 7]
 [3 3 3]
 [8 8 8]
 ...
 [9 9 9]
 [8 8 8]
 [1 1 1]]


In [15]:
blender = LogisticRegression(max_iter = 500, random_state = 42)
blender.fit(combined, y_val)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=500, random_state=42)

In [16]:
#Now to evaluate on the test set(Each model by themselves, in an ensemble, and finally with the blender)
scores = []
predictions = []
for model in fitted_models:
    preds = model.predict(X_test)
    predictions.append(preds)
    scores.append(accuracy_score(preds,y_test))

print("randomfrst, xtrees, svmc",scores)

preds = np.array(predictions).astype(int)
ensemble_preds = mode(preds, axis = 0).mode
print("ensemble", accuracy_score(ensemble_preds,y_test))



randomfrst, xtrees, svmc [0.9697, 0.9731, 0.9708]
ensemble 0.9741


In [17]:
a = np.array(predictions[0])
b = np.array(predictions[1])
c = np.array(predictions[2])
combined = np.column_stack((a,b,c))
print("With blender",accuracy_score(blender.predict(combined), y_test))



With blender 0.9593


So it seems that with the blender, the classifier actually performs worse. This might mean that the biases that the blender accounts for from the training data don't align with the test set.

In [18]:
#Sci kit learn stacking classifier
svm_pipeline.set_params(
    svc__C=10,
    svc__gamma=0.001,
    svc__kernel="rbf"
)
stack = StackingClassifier(estimators = [
    ("forest", RandomForestClassifier(max_depth= 20, max_features= "sqrt", min_samples_leaf= 1,
                                      min_samples_split= 2, n_estimators= 400, random_state = 42)),
    ("extratrees",ExtraTreesClassifier(max_depth= 30, max_features= "sqrt", min_samples_leaf= 1,
                                       min_samples_split= 2, n_estimators= 300, random_state = 42 )),
    ("SVMC",svm_pipeline)
    ],
    final_estimator=LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    stack_method="auto",
    cv=3,
    n_jobs=2,
    verbose=2)

In [19]:
stack.fit(X_train,y_train)
print("real stack", accuracy_score(stack.predict(X_test), y_test))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:  2.8min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:  4.6min finished
[Parallel(n_jobs=2)]: Done   3 out of   3 | elapsed:  9.3min finished


real stack 0.9782
